# Fase 02: Entrenamiento del Modelo de Forecasting (Decoder-Only Transformer)

Este notebook contiene la estructura y plantillas de código para el desarrollo, entrenamiento y validación del modelo de predicción temporal (forecasting) de temperatura basado en una arquitectura **Decoder-Only de Transformers**.

---

## 📋 Objetivos del Notebook
1. **Preprocesamiento Temporal (Windowing):** Transformar la serie temporal de temperaturas en secuencias/ventanas de entrenamiento (histórico -> futuro).
2. **Definición de la Arquitectura:** Diseñar un modelo autorregresivo (Decoder-Only) utilizando atención causal.
3. **Ciclo de Entrenamiento:** Configurar el bucle de optimización, función de pérdida (MSE/MAE) y parada temprana.
4. **Evaluación de Métricas:** Calcular métricas de predicción de series temporales (RMSE, MAE, MAPE).
5. **Inferencia Autoregresiva:** Implementar la lógica para predecir múltiples pasos hacia adelante (N días) que usará el endpoint `/forecast`.

---
## 🛠️ Configuración e Importación de Librerías

In [ ]:
# Importación de librerías para Deep Learning y Procesamiento
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Dispositivo de ejecución (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

## 1. Preparación del Dataset de Series Temporales (Windowing)

Para alimentar un modelo Transformer Decoder-only, necesitamos estructurar los datos en secuencias temporales consecutivas de longitud fija `(sequence_length)`.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len=72, pred_len=24):
        """
        Args:
            data: Array de NumPy o DataFrame con la variable objetivo escalada
            seq_len: Tamaño de la ventana histórica (ej. 72 horas)
            pred_len: Tamaño de la ventana a predecir (ej. 24 horas)
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        # Implementar tamaño máximo disponible basado en ventanas deslizantes
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        # Implementar extracción de x (histórico) e y (futuro)
        # Nota: Para decoder-only, y suele ser la misma secuencia desplazada o concatenada para autoregresión con máscara causal
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return x, y

In [ ]:
# Instanciar escalador, transformar los datos y crear DataLoader de entrenamiento y validación
scaler = MinMaxScaler(feature_range=(0, 1))

# dataset_train = TimeSeriesDataset(...)
# train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True)

## 2. Definición del Modelo (Decoder-Only Transformer)

Implementación del bloque Transformer con máscara causal para que el modelo no pueda ver los pasos futuros durante el entrenamiento autorregresivo.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # Implementar codificación posicional clásica sinoidal
        pass

    def forward(self, x):
        return x

class TemperatureDecoderOnly(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Capa Decoder de PyTorch
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.out_projection = nn.Linear(d_model, input_dim)
        
    def generate_causal_mask(self, sz):
        # Máscara triangular superior para impedir atención al futuro
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

    def forward(self, tgt):
        # tgt shape: [Batch_size, Seq_len, Input_dim]
        x = self.embedding(tgt)
        x = self.pos_encoder(x)
        
        mask = self.generate_causal_mask(tgt.size(1)).to(tgt.device)
        
        # Para decoder-only autoregresivo puro, llamamos pasándole la secuencia como target y memoria vacía
        # O se usa nn.TransformerEncoder con máscara causal
        out = self.transformer_decoder(tgt=x, memory=x, tgt_mask=mask) # Ajustar según diseño exacto
        preds = self.out_projection(out)
        return preds

model = TemperatureDecoderOnly().to(device)
print(model)

## 3. Bucle de Entrenamiento (Training Loop)

Definición de optimizador (AdamW), función de pérdida (MSELoss) y registro de métricas por época.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
epochs = 20

def train_model(model, train_loader, val_loader, epochs, criterion, optimizer):
    # Implementar entrenamiento y validación
    pass

# train_model(model, train_loader, val_loader, epochs, criterion, optimizer)

## 4. Inferencia Autoregresiva y Guardado del Modelo

Función generadora para predecir de forma recurrente (paso a paso), mapeando la salida predicha de vuelta a la entrada para la siguiente predicción.

In [ ]:
def predict_n_steps(model, seed_sequence, n_steps=24):
    """
    Lógica de autoregresión para predecir n_steps futuros uno a uno.
    """
    model.eval()
    preds = []
    current_seq = seed_sequence.clone()
    
    with torch.no_grad():
        for _ in range(n_steps):
            # Predecir el siguiente paso
            out = model(current_seq)
            next_pred = out[:, -1:, :] # Último elemento de la secuencia predicha
            preds.append(next_pred.cpu().numpy())
            # Actualizar secuencia deslizante
            current_seq = torch.cat([current_seq[:, 1:, :], next_pred], dim=1)
            
    return np.concatenate(preds, axis=1)

In [ ]:
# Código para guardar el modelo entrenado y los pesos del MinMax scaler (crucial para FastAPI)
# torch.save(model.state_index, '../models/transformer_forecast.pt')
# joblib.dump(scaler, '../models/scaler.pkl')